# 03 – Role and Pipeline Modeling

This notebook introduces modeled abstractions that transform raw job posting
information into decision-relevant features.

Specifically, we infer:
- Role families
- Pipeline levels
- Undergraduate accessibility

These abstractions enable downstream pipeline gap analysis and scenario modeling.

## Load Clean Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(".../Research Grant Project - Manufacturing Jobs - Manufacturing Information.csv")
df.shape

(388, 8)

## Role Family Inference

Job titles are mapped into high-level role families to reduce fragmentation
and enable aggregation across employers.

In [2]:
def infer_role_family(title: str) -> str:
    if pd.isna(title):
        return "Unknown"

    t = title.lower()

    if any(k in t for k in ["engineer", "engineering"]):
        return "Engineering"

    if any(k in t for k in ["technician", "operator", "manufacturing", "assembly", "production"]):
        return "Manufacturing Operations"

    if any(k in t for k in ["quality", "validation", "inspection", "compliance"]):
        return "Quality & Compliance"

    if any(k in t for k in ["supply", "logistics", "procurement", "materials"]):
        return "Supply Chain"

    if any(k in t for k in ["project", "program", "planner"]):
        return "Project Management"

    if any(k in t for k in ["design", "cad"]):
        return "Design"

    return "Other"


In [3]:
df["role_family"] = df["job_title"].map(infer_role_family)
df["role_family"].value_counts()

role_family
Engineering                 182
Other                       171
Project Management           17
Quality & Compliance          6
Supply Chain                  5
Manufacturing Operations      4
Design                        3
Name: count, dtype: int64

## Pipeline Level Inference

Pipeline level estimates how close a role is to the undergraduate talent pipeline.


In [5]:
def infer_pipeline_level(title: str) -> str:
    if pd.isna(title):
        return "Unknown"

    t = title.lower()

    if any(k in t for k in ["intern", "internship", "co-op", "coop"]):
        return "Internship"

    if any(k in t for k in ["entry", "junior", "associate", "i ", " i"]):
        return "Entry-Level"

    if any(k in t for k in ["senior", "sr", "lead", "principal", "manager", "director"]):
        return "Experienced"

    return "Unspecified"


In [7]:
df["pipeline_level"] = df["job_title"].map(infer_pipeline_level)
df["pipeline_level"].value_counts()

pipeline_level
Experienced    137
Internship     119
Unspecified    104
Entry-Level     28
Name: count, dtype: int64

## Undergraduate Accessibility Score

Accessibility is modeled as a heuristic score representing how suitable
a role is for undergraduate participation.

In [8]:
ACCESSIBILITY_MAP = {
    "Internship": 1.0,
    "Entry-Level": 0.7,
    "Unspecified": 0.4,
    "Experienced": 0.1,
    "Unknown": 0.0
}

df["accessibility_score"] = df["pipeline_level"].map(ACCESSIBILITY_MAP)
df["accessibility_score"].describe()

count    388.000000
mean       0.499742
std        0.372955
min        0.100000
25%        0.100000
50%        0.400000
75%        1.000000
max        1.000000
Name: accessibility_score, dtype: float64

## Accessible Demand by Role Family

This section compares total demand with undergraduate-accessible demand.

In [9]:
role_summary = (
    df.groupby("role_family")
      .agg(
          total_postings=("job_title", "count"),
          accessible_demand=("accessibility_score", "sum")
      )
      .sort_values("total_postings", ascending=False)
)

role_summary

,total_postings,accessible_demand
role_family,,
Engineering,182,95.0
Other,171,83.7
Project Management,17,7.4
Quality & Compliance,6,1.8
Supply Chain,5,3.5
Manufacturing Operations,4,1.9
Design,3,0.6


In [ ]:
OUT_PATH = ".../manufacturing_jobs_modeled.csv"
df.to_csv(OUT_PATH, index=False)
OUT_PATH

## Modeling Takeaways

- Manufacturing hiring demand spans multiple role families with varying levels of undergraduate accessibility.
- Engineering roles exhibit high demand but mixed accessibility.
- Internship and entry-level postings represent a minority of total demand.
- These patterns suggest targeted intervention opportunities rather than uniform investment.

These abstractions form the basis for pipeline gap and scenario analysis in the next notebook.